# ESCI reranker training (single-task baseline)

This notebook trains the original single-task cross-encoder reranker on ESCI (MSE loss on gains, nDCG eval). For the multi-task model (ranking + 4-class ESCI + substitute), see `train_multi_task_reranker.ipynb`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
CHECKPOINTS_DIR = ROOT / "checkpoints"

## Prepare train/val/test splits (run once)

Ensures `esci_train.parquet` and `esci_test.parquet` exist under `data/`. Training uses a validation split (held out by `query_id`) for mid-training eval; test is held out until final eval.

In [ ]:
from src.data.load_data import ESCIDataLoader
from src.data.utils import EXAMPLES_FILENAME

raw_data_dir = DATA_DIR if (DATA_DIR / EXAMPLES_FILENAME).exists() else None
loader = ESCIDataLoader(data_dir=raw_data_dir)
df = loader.load_esci(save_splits_dir=DATA_DIR)
print(f"Loaded {len(df)} rows; splits written under {DATA_DIR}")

## Configure and run training

In [ ]:
import logging
import yaml

from src.training.train_reranker import RerankerTrainer

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Load config from YAML (matches configs/reranker.yaml)
config_path = ROOT / "configs" / "reranker.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f) or {}

# Override paths for notebook (use absolute paths)
cfg["data_dir"] = str(DATA_DIR)
cfg["save_path"] = str(CHECKPOINTS_DIR / "reranker")
cfg.setdefault("max_length", 512)

# Pass only keys that RerankerTrainer accepts (exclude model_path, recall_at, etc.)
train_keys = {
    "data_dir", "model_name", "product_col", "save_path", "epochs", "batch_size",
    "max_length", "lr", "warmup_steps", "evaluation_steps", "early_stopping_patience", "val_frac",
    "eval_max_queries", "small_version", "device",
}
train_cfg = {k: v for k, v in cfg.items() if k in train_keys}
trainer = RerankerTrainer(**train_cfg)
model = trainer.run()